# EvalLLM 2026 RAG — Jupyter Notebook Ready Version



In [ ]:
# 第一次运行时取消注释：
# %pip install -q sentence-transformers numpy pandas requests tqdm

In [1]:
import json
import math
import os
import re
import sys
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

print("Imports OK")

Imports OK


In [2]:
WORKSPACE_DIR = Path(r"E:\CodexWorkspace\RAG_reading\RAG_0611")
PROJECT_ROOT = Path(r"E:\CodexWorkspace\RAG_reading")
DATA_DIR = PROJECT_ROOT / "donnee" / "Download 2026-04-30T08-00-29-368Z"

SAMPLE_QUERIES_PATH = WORKSPACE_DIR / "sample_queries.json"
CHUNKS_PATH = DATA_DIR / "chunks_for_rag.jsonl"
EMBEDDINGS_PATH = DATA_DIR / "chunk_embeddings(multi).npy"
METADATA_PATH = DATA_DIR / "chunk_metadata(multi).jsonl"

OUTPUT_JSON_PATH = WORKSPACE_DIR / "rag_evalllm2026_openrouter_answers_v3.json"
OUTPUT_MD_PATH = WORKSPACE_DIR / "rag_evalllm2026_openrouter_answers_v3.md"

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 64

# Official output can still keep 5 retrieved document/page entries, but the
# retriever now searches more broadly before selecting a diverse final context.
TOP_K = 5
CONTEXT_TOP_K = 10
HYBRID_CANDIDATES_K = 100
HYBRID_DENSE_WEIGHT = 0.45
HYBRID_BM25_WEIGHT = 0.55

BM25_K1 = 1.5
BM25_B = 0.75

# Multi-query retrieval improves coverage for questions that mention several
# entities, systems, countries, or acronyms.
ENABLE_MULTI_QUERY_RETRIEVAL = True
MAX_SUBQUERIES = 12
ENTITY_BONUS = 0.08
TITLE_MATCH_BONUS = 0.04
DIVERSITY_PENALTY = 0.10
MAX_CHUNKS_PER_DOC_PAGE = 1

# v3: exact anchor retrieval scans the local chunks for important named
# entities and domain phrases, then forces the best hits into the context.
ENABLE_EXACT_ANCHOR_RETRIEVAL = True
EXACT_ANCHOR_TOP_N = 2
EXACT_ANCHOR_MAX_TOTAL = 12
EXACT_ANCHOR_MIN_CHARS = 4
FORCED_ANCHOR_SCORE = 1.35

OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.5")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
# Set the key in your notebook/session before running generation:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
OPENROUTER_SITE_URL = os.getenv("OPENROUTER_SITE_URL", "")
OPENROUTER_APP_NAME = os.getenv("OPENROUTER_APP_NAME", "EvalLLM2026 RAG")

TEMPERATURE = 0
MAX_CONTEXT_CHARS_PER_CHUNK = 1800
MAX_RETRIES = 3
RETRY_SLEEP_SECONDS = 3

FORCE_REBUILD_EMBEDDINGS = False

# True = 不调用 OpenRouter，只用检索结果生成临时答案，适合先测试 pipeline
# False = 调用 OpenRouter，需要 API key
DRY_RUN_WITH_EXTRACTIVE_ANSWER = False

print("DATA_DIR:", DATA_DIR)
print("SAMPLE_QUERIES_PATH:", SAMPLE_QUERIES_PATH)
print("CHUNKS_PATH:", CHUNKS_PATH)
print("EMBEDDINGS_PATH:", EMBEDDINGS_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("OUTPUT_JSON_PATH:", OUTPUT_JSON_PATH)
print("OpenRouter model:", OPENROUTER_MODEL)


DATA_DIR: E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z
SAMPLE_QUERIES_PATH: E:\CodexWorkspace\RAG_reading\RAG_0611\sample_queries.json
CHUNKS_PATH: E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunks_for_rag.jsonl
EMBEDDINGS_PATH: E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunk_embeddings(multi).npy
METADATA_PATH: E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunk_metadata(multi).jsonl
OUTPUT_JSON_PATH: E:\CodexWorkspace\RAG_reading\RAG_0611\rag_evalllm2026_openrouter_answers_v3.json
OpenRouter model: openai/gpt-5.5


In [3]:
# 如果要调用 OpenRouter，把下面这一行取消注释并填入自己的 key：
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."

def get_openrouter_api_key() -> str:
    """
    在调用模型时动态读取 API key。
    这样你可以在 Notebook 中途设置 os.environ["OPENROUTER_API_KEY"]，
    不需要重新运行全部代码。
    """
    return os.getenv("OPENROUTER_API_KEY", "").strip()

print("OpenRouter API key set:", bool(get_openrouter_api_key()))

In [4]:
# 路径检查：这里只检查必须立即存在的输入文件
for path_name, path in {
    "SAMPLE_QUERIES_PATH": SAMPLE_QUERIES_PATH,
    "CHUNKS_PATH": CHUNKS_PATH,
}.items():
    if not path.exists():
        print(f"[WARNING] {path_name} does not exist: {path}")
    else:
        print(f"[OK] {path_name}: {path}")

[OK] SAMPLE_QUERIES_PATH: E:\CodexWorkspace\RAG_reading\RAG_0611\sample_queries.json
[OK] CHUNKS_PATH: E:\CodexWorkspace\RAG_reading\donnee\Download 2026-04-30T08-00-29-368Z\chunks_for_rag.jsonl


## 2. Load Queries

In [5]:
def load_queries(sample_queries_path: Path) -> list[dict[str, Any]]:
    with sample_queries_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    results = data.get("results", [])
    queries = []

    for idx, item in enumerate(results, 1):
        qid = item.get("qid") or f"Q{idx}"
        question = item.get("question")

        if not question:
            raise ValueError(f"Missing question for result item {idx}")

        queries.append(
            {
                "qid": qid,
                "question": question,
            }
        )

    return queries


queries = load_queries(SAMPLE_QUERIES_PATH)
print(f"Loaded queries: {len(queries)}")
pd.DataFrame(queries).head()

Loaded queries: 5


,qid,question
0,Q1,"Quel est l'objectif du projet ""Beehive"" de la ..."
1,Q2,Comment l’intégration du drone MQ-9 Reaper dan...
2,Q3,Comment les mesures anti-drones proposées en F...
3,Q4,Comment les méthodes de renseignement telles q...
4,Q5,Quels sont les drones capables de transporter ...


## 3. Load or Build Embedding Index

In [6]:
def load_chunks(chunks_path: Path) -> list[dict[str, Any]]:
    chunks = []

    with chunks_path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue

            item = json.loads(line)

            missing = [
                key
                for key in ["chunk_id", "doc_name", "page", "text"]
                if key not in item
            ]
            if missing:
                raise ValueError(
                    f"Line {line_no} is missing required fields: {missing}"
                )

            chunks.append(item)

    return chunks


def save_metadata(chunks: list[dict[str, Any]], metadata_path: Path) -> None:
    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    with metadata_path.open("w", encoding="utf-8") as f:
        for chunk in chunks:
            item = {
                "chunk_id": chunk["chunk_id"],
                "doc_name": chunk["doc_name"],
                "page": chunk["page"],
                "text": chunk["text"],
                "word_count": chunk.get("word_count"),
                "char_count": chunk.get("char_count"),
                "source_type": chunk.get("source_type"),
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def load_metadata(metadata_path: Path) -> list[dict[str, Any]]:
    metadata = []

    with metadata_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                metadata.append(json.loads(line))

    return metadata


def build_or_load_embedding_index(
    chunks_path: Path = CHUNKS_PATH,
    embeddings_path: Path = EMBEDDINGS_PATH,
    metadata_path: Path = METADATA_PATH,
    model_name: str = EMBEDDING_MODEL_NAME,
    batch_size: int = BATCH_SIZE,
    force_rebuild: bool = FORCE_REBUILD_EMBEDDINGS,
):
    if embeddings_path.exists() and metadata_path.exists() and not force_rebuild:
        print("Loading existing embedding index.")
        embeddings = np.load(embeddings_path)
        metadata = load_metadata(metadata_path)
        model = SentenceTransformer(model_name)

        if len(metadata) != len(embeddings):
            raise ValueError(
                f"Metadata count ({len(metadata)}) does not match "
                f"embedding count ({len(embeddings)})."
            )

        return model, embeddings, metadata

    if not chunks_path.exists():
        raise FileNotFoundError(f"Missing chunks file: {chunks_path}")

    print("Building embedding index from chunks.")
    chunks = load_chunks(chunks_path)
    texts = [chunk["text"] for chunk in chunks]

    model = SentenceTransformer(model_name)

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    embeddings = np.asarray(embeddings, dtype="float32")

    embeddings_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(embeddings_path, embeddings)
    save_metadata(chunks, metadata_path)

    metadata = load_metadata(metadata_path)
    return model, embeddings, metadata


embedding_model, embeddings, metadata = build_or_load_embedding_index()

print(f"Metadata records: {len(metadata)}")
print(f"Embedding shape: {embeddings.shape}")

pd.DataFrame(metadata).head()

Loading existing embedding index.
Metadata records: 29202
Embedding shape: (29202, 384)


,chunk_id,doc_name,page,text,word_count,char_count,source_type
0,04500463_rev_e_smdr_feature_ovvu_p1_c000,04500463_rev_e_smdr_feature_ovvu.pdf,1,"ESI (Estech Systems, Inc.) • 800 374-0422 • fa...",200,1263,pymupdf
1,04500463_rev_e_smdr_feature_ovvu_p1_c001,04500463_rev_e_smdr_feature_ovvu.pdf,1,the ESI Applications Services Card (ASC) or (b...,160,1119,pymupdf
2,04500463_rev_e_smdr_feature_ovvu_p1_c002,04500463_rev_e_smdr_feature_ovvu.pdf,1,XXXXXXXXXXXXXXXXXXXXXXXXXXX AAAAAAAAAA EE RR L...,10,79,pymupdf
3,04500463_rev_e_smdr_feature_ovvu_p2_c000,04500463_rev_e_smdr_feature_ovvu.pdf,2,SMDR Feature Overview 2 The columns are: • Cal...,200,1185,pymupdf
4,04500463_rev_e_smdr_feature_ovvu_p2_c001,04500463_rev_e_smdr_feature_ovvu.pdf,2,Begins at column 77. Each record is terminated...,200,1481,pymupdf


## 4. Dense Retrieval + BM25 Retrieval + Hybrid Retrieval

In [7]:
def get_text(item: dict[str, Any]) -> str:
    return item.get("text") or item.get("texte") or ""


def get_doc_name(item: dict[str, Any]) -> str:
    return item.get("doc_name") or item.get("source") or ""


def get_page(item: dict[str, Any]) -> int | None:
    page = item.get("page")

    if page is None:
        return None

    try:
        return int(page)
    except (TypeError, ValueError):
        return page


def get_chunk_id(item: dict[str, Any], idx: int | None = None) -> str:
    if "chunk_id" in item:
        return str(item["chunk_id"])

    return f"{get_doc_name(item)}_page_{get_page(item)}_idx_{idx}"


def tokenize_for_bm25(text: str) -> list[str]:
    """
    Unicode tokenizer.
    可以处理英文、法文、数字、部分带连字符的词。
    对中文语料来说，BM25 效果取决于文本是否已经分词；
    如果文本主要是中文，可之后换成 jieba 或更适合的 tokenizer。
    """
    return re.findall(r"(?u)\b\w+(?:[-']\w+)*\b", text.lower())


def build_bm25_index(metadata: list[dict[str, Any]]) -> dict[str, Any]:
    doc_term_freqs = []
    doc_lengths = []
    doc_freqs = defaultdict(int)

    for item in metadata:
        doc_text = f"{get_doc_name(item)} {get_text(item)}"
        term_freqs = Counter(tokenize_for_bm25(doc_text))
        doc_term_freqs.append(term_freqs)
        doc_lengths.append(sum(term_freqs.values()))

        for term in term_freqs:
            doc_freqs[term] += 1

    doc_count = len(metadata)
    avg_doc_length = sum(doc_lengths) / doc_count if doc_count else 0.0

    idf = {
        term: math.log(1 + (doc_count - freq + 0.5) / (freq + 0.5))
        for term, freq in doc_freqs.items()
    }

    inverted_index = defaultdict(list)

    for doc_idx, term_freqs in enumerate(doc_term_freqs):
        for term, freq in term_freqs.items():
            inverted_index[term].append((doc_idx, freq))

    return {
        "doc_count": doc_count,
        "avg_doc_length": avg_doc_length,
        "doc_lengths": doc_lengths,
        "idf": idf,
        "inverted_index": inverted_index,
        "k1": BM25_K1,
        "b": BM25_B,
    }


def dense_retrieve(query: str, top_k: int) -> list[dict[str, Any]]:
    query_embedding = embedding_model.encode(query, normalize_embeddings=True)
    query_embedding = np.asarray(query_embedding, dtype="float32")

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, 1):
        idx = int(idx)
        item = metadata[idx]

        results.append(
            {
                "rank": rank,
                "score": float(scores[idx]),
                "chunk_id": get_chunk_id(item, idx),
                "doc_name": get_doc_name(item),
                "page": get_page(item),
                "text": get_text(item),
            }
        )

    return results


def bm25_retrieve(
    query: str,
    bm25_index: dict[str, Any],
    top_k: int,
) -> list[dict[str, Any]]:
    query_terms = tokenize_for_bm25(query)
    scores = defaultdict(float)
    avg_doc_length = bm25_index["avg_doc_length"]

    if not query_terms or not avg_doc_length:
        return []

    for term in query_terms:
        term_idf = bm25_index["idf"].get(term)

        if term_idf is None:
            continue

        for doc_idx, term_freq in bm25_index["inverted_index"].get(term, []):
            doc_length = bm25_index["doc_lengths"][doc_idx]

            denominator = term_freq + bm25_index["k1"] * (
                1
                - bm25_index["b"]
                + bm25_index["b"] * doc_length / avg_doc_length
            )

            scores[doc_idx] += term_idf * (
                term_freq * (bm25_index["k1"] + 1) / denominator
            )

    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]
    results = []

    for rank, (idx, score) in enumerate(ranked, 1):
        idx = int(idx)
        item = metadata[idx]

        results.append(
            {
                "rank": rank,
                "score": float(score),
                "chunk_id": get_chunk_id(item, idx),
                "doc_name": get_doc_name(item),
                "page": get_page(item),
                "text": get_text(item),
            }
        )

    return results


def normalize_score_map(results: list[dict[str, Any]]) -> dict[str, float]:
    if not results:
        return {}

    scores = [item["score"] for item in results]
    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:
        return {item["chunk_id"]: 1.0 for item in results}

    return {
        item["chunk_id"]: (item["score"] - min_score) / (max_score - min_score)
        for item in results
    }


def normalize_for_match(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("’", "'").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text.lower()).strip()


def dedupe_keep_order(items: list[str]) -> list[str]:
    seen = set()
    deduped = []

    for item in items:
        item = re.sub(r"\s+", " ", item).strip(" ,.;:()[]{}")
        if not item:
            continue

        key = normalize_for_match(item)
        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    return deduped


def extract_query_terms(question: str) -> list[str]:
    """Extract high-precision named anchors from the question."""
    candidates = []
    candidates.extend(re.findall(r'"([^"]+)"', question))
    candidates.extend(re.findall(r"\b[A-ZÀ-ÖØ-Þ0-9][A-ZÀ-ÖØ-Þ0-9-]{2,}\b", question))
    candidates.extend(re.findall(r"\b[A-ZÀ-ÖØ-Þ][\wÀ-ÿ'’-]+(?:\s+[A-Z0-9][\wÀ-ÿ'’-]+){1,3}\b", question))
    candidates.extend(re.findall(r"\b[A-ZÀ-ÖØ-Þ]+-\d+[A-Z0-9-]*\b", question))
    candidates.extend(re.findall(r"\b\d+[A-Z]-[A-Z]\b", question))

    important_lower = [
        "tracfin",
        "brouillage",
        "anti-drones",
        "anti-drone",
        "guerre électronique",
        "munitions rôdeuses",
        "stratégie navale chinoise",
        "usv ukrainiens",
        "drones navals ukrainiens",
    ]
    q_norm = normalize_for_match(question)
    for phrase in important_lower:
        if normalize_for_match(phrase) in q_norm:
            candidates.append(phrase)

    stop = {
        "Comment", "Quel", "Quels", "Quelle", "Quelles", "Pour", "Dans",
        "French", "The", "Les", "Des", "Ministère",
    }
    cleaned = []

    for candidate in dedupe_keep_order(candidates):
        if len(candidate) < EXACT_ANCHOR_MIN_CHARS:
            continue
        if candidate in stop:
            continue
        cleaned.append(candidate)

    return cleaned[:MAX_SUBQUERIES]


def expand_domain_anchors(question: str, anchors: list[str]) -> list[str]:
    q = normalize_for_match(question)
    expanded = list(anchors)

    if "tekever" in q or "ar-3" in q:
        expanded.extend(["TEKEVER AR-3", "AR-3", "AR-3-TEKEVER-POR", "XPONENTIAL 2023 TEKEVER AR-3"])
    if "tracfin" in q:
        expanded.extend(["Tracfin", "l16b1454 Tracfin", "financement étranger associations cultuelles", "financement étranger lieux de culte"])
    if "brouillage" in q or "anti-drone" in q or "anti-drones" in q or "neutralisation" in q:
        expanded.extend(["brouillage des drones", "lutte anti-drone", "neutralisation des drones", "l15b4320 brouillage drones", "autoriser recours dispositifs brouillage drones menace imminente"])
    if "ukrain" in q and "usv" in q:
        expanded.extend(["USV ukrainiens", "attaques d'USV ukrainiens", "drones navals ukrainiens"])
    if "chine" in q or "chinoise" in q or "chinois" in q:
        expanded.extend(["Chine USV ukrainiens", "stratégie chinoise drones navals", "marine chinoise USV"])
    if "altius" in q:
        expanded.extend(["Altius 600M-V", "Anduril Altius 600M-V"])
    if "switchblade" in q:
        expanded.extend(["Switchblade 300", "AeroVironment Switchblade 300"])

    return dedupe_keep_order(expanded)[:MAX_SUBQUERIES + 12]


def build_subqueries(question: str) -> list[str]:
    anchors = expand_domain_anchors(question, extract_query_terms(question))
    subqueries = [question]

    for anchor in anchors:
        subqueries.append(anchor)
        subqueries.append(f"{anchor} {question}")

    return dedupe_keep_order(subqueries)[: 1 + 2 * len(anchors)]


def coverage_bonus(question_terms: list[str], item: dict[str, Any]) -> float:
    haystack = normalize_for_match(f"{item['doc_name']} {item['text']}")
    doc_name = normalize_for_match(item["doc_name"])
    bonus = 0.0

    for term in question_terms:
        term_l = normalize_for_match(term)
        if term_l in haystack:
            bonus += ENTITY_BONUS
        if term_l in doc_name:
            bonus += TITLE_MATCH_BONUS

    return bonus


def build_anchor_search_index(metadata: list[dict[str, Any]]) -> list[dict[str, Any]]:
    index = []

    for idx, item in enumerate(metadata):
        haystack = normalize_for_match(f"{get_doc_name(item)} {get_text(item)}")
        index.append(
            {
                "idx": idx,
                "item": item,
                "haystack": haystack,
                "terms": set(tokenize_for_bm25(haystack)),
                "doc_name_norm": normalize_for_match(get_doc_name(item)),
            }
        )

    return index


def exact_anchor_retrieve(
    question: str,
    anchors: list[str],
    max_total: int = EXACT_ANCHOR_MAX_TOTAL,
    top_n_per_anchor: int = EXACT_ANCHOR_TOP_N,
) -> list[dict[str, Any]]:
    if not ENABLE_EXACT_ANCHOR_RETRIEVAL:
        return []

    search_index = globals().get("anchor_search_index")
    if search_index is None:
        search_index = build_anchor_search_index(metadata)

    question_terms = {
        tok
        for tok in tokenize_for_bm25(normalize_for_match(question))
        if len(tok) > 2
    }
    anchor_hits = []

    for anchor in anchors:
        anchor_norm = normalize_for_match(anchor)
        if len(anchor_norm) < EXACT_ANCHOR_MIN_CHARS:
            continue

        anchor_terms = set(tokenize_for_bm25(anchor_norm))
        if not anchor_terms:
            continue

        scored = []

        for rec in search_index:
            phrase_match = anchor_norm in rec["haystack"]
            term_match = anchor_terms.issubset(rec["terms"])
            if not phrase_match and not term_match:
                continue

            item = rec["item"]
            overlap = len(question_terms & rec["terms"])
            title_match = 1 if anchor_norm in rec["doc_name_norm"] else 0
            exact_count = rec["haystack"].count(anchor_norm) if phrase_match else 0
            score = FORCED_ANCHOR_SCORE + 0.04 * overlap + 0.15 * title_match + 0.12 * exact_count
            if "brouillage" in anchor_norm and "menace imminente" in rec["haystack"]:
                score += 0.65
            if "brouillage" in anchor_norm and "services de l'etat" in rec["haystack"]:
                score += 0.35
            if "tracfin" in anchor_norm and "associations cultuelles" in rec["haystack"]:
                score += 0.45

            scored.append(
                {
                    "rank": 0,
                    "score": float(score),
                    "dense_score": 0.0,
                    "bm25_score": 0.0,
                    "chunk_id": get_chunk_id(item, rec["idx"]),
                    "doc_name": get_doc_name(item),
                    "page": get_page(item),
                    "text": get_text(item),
                    "retrieval_sources": ["exact_anchor"],
                    "matched_subqueries": [anchor],
                }
            )

        scored.sort(key=lambda item: item["score"], reverse=True)
        anchor_hits.extend(scored[:top_n_per_anchor])

    deduped = {}
    for item in anchor_hits:
        key = item["chunk_id"]
        if key not in deduped or item["score"] > deduped[key]["score"]:
            deduped[key] = item

    ranked = sorted(deduped.values(), key=lambda item: item["score"], reverse=True)
    return ranked[:max_total]


def hybrid_retrieve_single_query(
    query: str,
    bm25_index: dict[str, Any],
    candidate_k: int = HYBRID_CANDIDATES_K,
    dense_weight: float = HYBRID_DENSE_WEIGHT,
    bm25_weight: float = HYBRID_BM25_WEIGHT,
) -> list[dict[str, Any]]:
    dense_results = dense_retrieve(query, top_k=candidate_k)
    bm25_results = bm25_retrieve(query, bm25_index=bm25_index, top_k=candidate_k)

    dense_scores = normalize_score_map(dense_results)
    bm25_scores = normalize_score_map(bm25_results)

    merged = {}

    for source_name, source_results in [
        ("dense", dense_results),
        ("bm25", bm25_results),
    ]:
        for item in source_results:
            chunk_id = item["chunk_id"]

            if chunk_id not in merged:
                merged[chunk_id] = {
                    "chunk_id": chunk_id,
                    "doc_name": item["doc_name"],
                    "page": item["page"],
                    "text": item["text"],
                    "retrieval_sources": [],
                }

            merged[chunk_id]["retrieval_sources"].append(source_name)

    ranked = []

    for chunk_id, item in merged.items():
        dense_score = dense_scores.get(chunk_id, 0.0)
        bm25_score = bm25_scores.get(chunk_id, 0.0)
        final_score = dense_weight * dense_score + bm25_weight * bm25_score

        ranked.append(
            {
                "score": float(final_score),
                "dense_score": float(dense_score),
                "bm25_score": float(bm25_score),
                **item,
            }
        )

    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked


def hybrid_retrieve(
    query: str,
    bm25_index: dict[str, Any],
    top_k: int = TOP_K,
    candidate_k: int = HYBRID_CANDIDATES_K,
    dense_weight: float = HYBRID_DENSE_WEIGHT,
    bm25_weight: float = HYBRID_BM25_WEIGHT,
) -> list[dict[str, Any]]:
    if not ENABLE_MULTI_QUERY_RETRIEVAL:
        ranked = hybrid_retrieve_single_query(
            query,
            bm25_index=bm25_index,
            candidate_k=candidate_k,
            dense_weight=dense_weight,
            bm25_weight=bm25_weight,
        )[:top_k]
        for rank, item in enumerate(ranked, 1):
            item["rank"] = rank
        return ranked

    base_terms = extract_query_terms(query)
    question_terms = expand_domain_anchors(query, base_terms)
    subqueries = build_subqueries(query)
    merged = {}

    for item in exact_anchor_retrieve(query, question_terms):
        merged[item["chunk_id"]] = {
            **item,
            "retrieval_sources": set(item["retrieval_sources"]),
            "matched_subqueries": list(item.get("matched_subqueries", [])),
        }

    for subquery_idx, subquery in enumerate(subqueries):
        sub_results = hybrid_retrieve_single_query(
            subquery,
            bm25_index=bm25_index,
            candidate_k=candidate_k,
            dense_weight=dense_weight,
            bm25_weight=bm25_weight,
        )

        weight = 1.0 if subquery_idx == 0 else 0.72

        for rank_idx, item in enumerate(sub_results, 1):
            chunk_id = item["chunk_id"]
            rank_bonus = 1.0 / (rank_idx + 60)
            weighted_score = weight * item["score"] + rank_bonus

            if chunk_id not in merged:
                merged[chunk_id] = {
                    **item,
                    "score": 0.0,
                    "dense_score": 0.0,
                    "bm25_score": 0.0,
                    "retrieval_sources": set(),
                    "matched_subqueries": [],
                }

            merged_item = merged[chunk_id]
            merged_item["score"] = max(merged_item["score"], weighted_score)
            merged_item["dense_score"] = max(merged_item["dense_score"], item["dense_score"])
            merged_item["bm25_score"] = max(merged_item["bm25_score"], item["bm25_score"])
            if not isinstance(merged_item["retrieval_sources"], set):
                merged_item["retrieval_sources"] = set(merged_item["retrieval_sources"])
            merged_item["retrieval_sources"].update(item["retrieval_sources"])

            if len(merged_item["matched_subqueries"]) < 4:
                merged_item["matched_subqueries"].append(subquery)

    ranked = []
    for item in merged.values():
        item["score"] += coverage_bonus(question_terms, item)
        item["retrieval_sources"] = sorted(item["retrieval_sources"])
        ranked.append(item)

    ranked.sort(key=lambda item: item["score"], reverse=True)

    selected = []
    doc_page_counts = defaultdict(int)
    selected_docs = set()

    for item in ranked:
        key = (item["doc_name"], item["page"])
        if doc_page_counts[key] >= MAX_CHUNKS_PER_DOC_PAGE:
            continue

        adjusted_score = item["score"]
        if item["doc_name"] in selected_docs:
            adjusted_score -= DIVERSITY_PENALTY

        item = {**item, "score": adjusted_score}
        selected.append(item)
        doc_page_counts[key] += 1
        selected_docs.add(item["doc_name"])

        if len(selected) >= top_k:
            break

    selected.sort(key=lambda item: item["score"], reverse=True)

    for rank, item in enumerate(selected, 1):
        item["rank"] = rank

    return selected


bm25_index = build_bm25_index(metadata)
anchor_search_index = build_anchor_search_index(metadata)
print(f"BM25 documents: {bm25_index['doc_count']}")
print(f"Anchor search records: {len(anchor_search_index)}")


BM25 documents: 29202
Anchor search records: 29202


## 5. Test One Query

先测试一个问题，确认检索结果是否正常。

In [8]:
test_query = queries[0]["question"]
test_results = hybrid_retrieve(test_query, bm25_index=bm25_index, top_k=5)

print("Question:", test_query)
print()

for item in test_results:
    preview = " ".join(item["text"].split())[:300]
    print(
        f"rank={item['rank']} "
        f"score={item['score']:.4f} "
        f"dense={item['dense_score']:.4f} "
        f"bm25={item['bm25_score']:.4f} | "
        f"{item['doc_name']} p.{item['page']}"
    )
    print(preview)
    print("-" * 80)

Question: Quel est l'objectif du projet "Beehive" de la Royal Navy, et quelles sont les caractéristiques clés des USV (véhicules de surface sans équipage) qu'elle souhaite acquérir dans ce cadre ?

rank=1 score=2.4700 dense=0.0000 bm25=1.0000 | 20260108_NP_Obsdrones_Bulletin-de-veille-n12_0.pdf p.17
Le Corsair ASV a une CU de 453 kg sur 1 000 nq. Le Corsair ASV est long de 7.32 m et a une vitesse de 35 nds. Aujourd’hui, Saronic propose une gamme d’USV allant du Spyglass de 1,83 m, jusqu’au Marauder de 54.9 m de long. La Royal Navy chercherait à acquérir 20 USV Source : UK Defence Journal Analys
--------------------------------------------------------------------------------
rank=2 score=2.1500 dense=1.0000 bm25=0.7723 | COLSbleus_3126_PDFweb_PAGE.pdf p.21
DES MARINS français SUR DES NAVIRES DE LA ROYAL NAVY E n 2015, dans le cadre des échanges réguliers entre le vice-amiral d’escadre Jonathan Woodcock, (Second Sea Lord), et le vice-amiral d’escadre Christopje Prazuk (directeur de perso

## 6. Official JSON Helpers

In [9]:
def official_retrieved_items(
    retrieved_chunks: list[dict[str, Any]],
    max_items: int = TOP_K,
) -> list[dict[str, Any]]:
    """
    输出官方需要的 retrieved 结构：
    - rank
    - doc_name
    - page
    - metadata
    """
    seen = set()
    official = []

    for item in retrieved_chunks:
        key = (item["doc_name"], item["page"])

        if key in seen:
            continue

        seen.add(key)

        official.append(
            {
                "rank": len(official) + 1,
                "doc_name": item["doc_name"],
                "page": item["page"],
                "metadata": {
                    "chunk_id": item["chunk_id"],
                    "score": round(float(item["score"]), 6),
                    "dense_score": round(float(item["dense_score"]), 6),
                    "bm25_score": round(float(item["bm25_score"]), 6),
                    "retrieval_sources": item["retrieval_sources"],
                    "matched_subqueries": item.get("matched_subqueries", [])[:3],
                },
            }
        )

        if len(official) >= max_items:
            break

    return official


def build_context(retrieved_chunks: list[dict[str, Any]]) -> str:
    blocks = []

    for item in retrieved_chunks:
        text = " ".join(item["text"].split())

        if len(text) > MAX_CONTEXT_CHARS_PER_CHUNK:
            text = text[:MAX_CONTEXT_CHARS_PER_CHUNK].rstrip() + "..."

        matched = "; ".join(item.get("matched_subqueries", [])[:3])
        blocks.append(
            "\n".join(
                [
                    f"[{item['rank']}] doc_name: {item['doc_name']}",
                    f"page: {item['page']}",
                    f"chunk_id: {item['chunk_id']}",
                    f"matched_subqueries: {matched}",
                    f"text: {text}",
                ]
            )
        )

    return "\n\n".join(blocks)


def fallback_extractive_answer(
    question: str,
    retrieved_chunks: list[dict[str, Any]],
) -> str:
    """
    不调用 LLM 时的临时答案。
    用于测试 pipeline 是否能跑通。
    """
    if not retrieved_chunks:
        return "Aucun passage pertinent n'a été retrouvé dans la collection fournie."

    lines = [
        "Réponse extractive provisoire fondée uniquement sur les passages récupérés.",
        f"Question: {question}",
        "",
    ]

    for item in retrieved_chunks[:3]:
        preview = " ".join(item["text"].split())[:700]
        lines.append(f"- {item['doc_name']}, page {item['page']}: {preview}")

    return "\n".join(lines)


## 7. OpenRouter Generation

In [10]:
def call_openrouter(prompt: str, model: str = OPENROUTER_MODEL) -> str:
    api_key = get_openrouter_api_key()

    if not api_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set. "
            "Set os.environ['OPENROUTER_API_KEY'] first, "
            "or set DRY_RUN_WITH_EXTRACTIVE_ANSWER = True."
        )

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    if OPENROUTER_SITE_URL:
        headers["HTTP-Referer"] = OPENROUTER_SITE_URL

    if OPENROUTER_APP_NAME:
        headers["X-Title"] = OPENROUTER_APP_NAME

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Tu es un système RAG pour le challenge EvalLLM 2026. "
                    "Réponds en français, uniquement avec les informations fournies "
                    "dans le contexte. Couvre explicitement chaque entité ou sous-question "
                    "présente dans la question. Si une entité n'est pas documentée par les "
                    "passages, dis-le clairement au lieu d'inventer."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        "temperature": TEMPERATURE,
    }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                OPENROUTER_BASE_URL,
                headers=headers,
                json=payload,
                timeout=120,
            )

            if (
                response.status_code in {429, 500, 502, 503, 504}
                and attempt < MAX_RETRIES
            ):
                time.sleep(RETRY_SLEEP_SECONDS * attempt)
                continue

            response.raise_for_status()
            data = response.json()

            return data["choices"][0]["message"]["content"].strip()

        except Exception as exc:
            last_error = exc

            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP_SECONDS * attempt)
                continue

    raise RuntimeError(
        f"OpenRouter call failed after {MAX_RETRIES} attempts: {last_error}"
    )


def generate_answer(
    question: str,
    retrieved_chunks: list[dict[str, Any]],
) -> str:
    if DRY_RUN_WITH_EXTRACTIVE_ANSWER:
        return fallback_extractive_answer(question, retrieved_chunks)

    context = build_context(retrieved_chunks)
    anchors = expand_domain_anchors(question, extract_query_terms(question))
    anchors_text = ", ".join(anchors) if anchors else "aucune entité extraite"

    prompt = f"""Question:
{question}

Entités/sous-thèmes à couvrir si les sources le permettent:
{anchors_text}

Passages récupérés:
{context}

Consignes:
- Rédige une réponse pertinente et complète en français.
- N'utilise aucune information externe aux passages récupérés.
- Couvre séparément les entités, pays, systèmes ou capacités nommés dans la question.
- Pour une question comparative ou multi-parties, réponds à chaque partie avant la synthèse.
- Ne cite pas de sources inventées.
- Si un élément demandé n'est pas établi par les passages, indique précisément lequel manque.
- Garde une réponse concise, structurée en paragraphes ou puces si utile.
"""

    return call_openrouter(prompt)


## 8. Run Full Pipeline

In [11]:
def run_pipeline(
    queries: list[dict[str, Any]],
    output_json_path: Path = OUTPUT_JSON_PATH,
    output_md_path: Path = OUTPUT_MD_PATH,
    top_k: int = TOP_K,
    context_top_k: int = CONTEXT_TOP_K,
    candidate_k: int = HYBRID_CANDIDATES_K,
) -> dict[str, Any]:
    output = {
        "run_id": f"evalllm2026_hybrid_openrouter_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        "parameters": {
            "retriever_version": 5,
            "retriever_type": "anchor_coverage_multi_query_hybrid_dense_bm25",
            "embedding_model_name": EMBEDDING_MODEL_NAME,
            "model_name": OPENROUTER_MODEL,
            "temperature": TEMPERATURE,
            "top_k": top_k,
            "context_top_k": context_top_k,
            "candidate_k_each": candidate_k,
            "dense_weight": HYBRID_DENSE_WEIGHT,
            "bm25_weight": HYBRID_BM25_WEIGHT,
            "bm25_k1": BM25_K1,
            "bm25_b": BM25_B,
            "multi_query": ENABLE_MULTI_QUERY_RETRIEVAL,
            "max_subqueries": MAX_SUBQUERIES,
            "exact_anchor_retrieval": ENABLE_EXACT_ANCHOR_RETRIEVAL,
            "exact_anchor_top_n": EXACT_ANCHOR_TOP_N,
            "entity_bonus": ENTITY_BONUS,
            "title_match_bonus": TITLE_MATCH_BONUS,
            "no_web_search": True,
            "dry_run": DRY_RUN_WITH_EXTRACTIVE_ANSWER,
        },
        "results": [],
    }

    md_lines = [
        "# EvalLLM 2026 RAG Answers",
        "",
        f"Generated at: {datetime.now().isoformat(timespec='seconds')}",
        f"OpenRouter model: {OPENROUTER_MODEL}",
        (
            "Retriever: multi-query hybrid dense + BM25, "
            f"top_k={top_k}, context_top_k={context_top_k}, candidate_k={candidate_k}"
        ),
        f"Dry run: {DRY_RUN_WITH_EXTRACTIVE_ANSWER}",
        "",
    ]

    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    output_md_path.parent.mkdir(parents=True, exist_ok=True)

    for query in tqdm(queries, desc="RAG queries"):
        retrieved_chunks = hybrid_retrieve(
            query["question"],
            bm25_index=bm25_index,
            top_k=context_top_k,
            candidate_k=candidate_k,
        )

        answer = generate_answer(query["question"], retrieved_chunks)
        retrieved = official_retrieved_items(retrieved_chunks, max_items=top_k)

        output["results"].append(
            {
                "qid": query["qid"],
                "question": query["question"],
                "retrieved": retrieved,
                "answer": answer,
                "metadata": {
                    "generation_model": OPENROUTER_MODEL,
                    "retrieval_method": "anchor_coverage_multi_query_hybrid_dense_bm25",
                    "context_chunk_count": len(retrieved_chunks),
                    "query_terms": expand_domain_anchors(query["question"], extract_query_terms(query["question"])),
                    "generated_at": datetime.now().isoformat(timespec="seconds"),
                },
            }
        )

        md_lines.extend(
            [
                f"## {query['qid']}",
                "",
                f"**Question:** {query['question']}",
                "",
                "**Retrieved:**",
            ]
        )

        for item in retrieved:
            md_lines.append(
                f"- rank {item['rank']}: {item['doc_name']} page {item['page']}"
            )

        md_lines.extend(
            [
                "",
                "**Answer:**",
                "",
                answer,
                "",
            ]
        )

    with output_json_path.open("w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    output_md_path.write_text("\n".join(md_lines), encoding="utf-8")

    print(f"Saved official JSON: {output_json_path}")
    print(f"Saved readable Markdown: {output_md_path}")

    return output


In [12]:
import os
import time
import requests
from getpass import getpass

# 第一次运行时输入你的 Mistral 官方 API key
# 注意：这里不是 OpenRouter key，而是 Mistral API key
if not os.getenv("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass("Paste your Mistral API key: ")

# 可以先用 small，便宜且够用；如果答案质量不够，再换 large
#MISTRAL_MODEL = "mistral-small-latest"
MISTRAL_MODEL = "mistral-large-latest"

MISTRAL_API_URL = "https://api.mistral.ai/v1/chat/completions"


def call_mistral(prompt: str, model: str | None = None) -> str:
    api_key = os.getenv("MISTRAL_API_KEY")

    if not api_key:
        raise RuntimeError("MISTRAL_API_KEY is missing.")

    if model is None:
        model = MISTRAL_MODEL

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ],
        "temperature": 0.2,
        "max_tokens": 1200,
    }

    max_retries = globals().get("MAX_RETRIES", 3)
    retry_sleep_seconds = globals().get("RETRY_SLEEP_SECONDS", 2)
    request_timeout = globals().get("REQUEST_TIMEOUT", 120)

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                MISTRAL_API_URL,
                headers=headers,
                json=payload,
                timeout=request_timeout,
            )

            if response.status_code == 401:
                raise RuntimeError(
                    "Mistral 401 Unauthorized: MISTRAL_API_KEY is missing, invalid, "
                    "or not correctly passed."
                )

            response.raise_for_status()

            data = response.json()
            return data["choices"][0]["message"]["content"]

        except Exception as e:
            last_error = e
            print(f"[Mistral attempt {attempt}] {repr(e)}")

            if attempt < max_retries:
                time.sleep(retry_sleep_seconds * attempt)

    raise RuntimeError(
        f"Mistral call failed after {max_retries} attempts: {last_error}"
    )


# 关键：保留原函数名，让 generate_answer() 不用改
call_openrouter = call_mistral

Paste your Mistral API key:  ········


In [13]:
rag_output = run_pipeline(queries)

summary_df = pd.DataFrame(
    [
        {
            "qid": item["qid"],
            "retrieved_count": len(item["retrieved"]),
            "answer_chars": len(item["answer"]),
        }
        for item in rag_output["results"]
    ]
)

summary_df

RAG queries:   0%|          | 0/5 [00:00<?, ?it/s]

Saved official JSON: E:\CodexWorkspace\RAG_reading\RAG_0611\rag_evalllm2026_openrouter_answers_v3.json
Saved readable Markdown: E:\CodexWorkspace\RAG_reading\RAG_0611\rag_evalllm2026_openrouter_answers_v3.md


,qid,retrieved_count,answer_chars
0,Q1,5,2232
1,Q2,5,4582
2,Q3,5,4363
3,Q4,5,5075
4,Q5,5,4294


## 9. Validate Output Shape

In [14]:
def validate_official_task1_json(output: dict[str, Any]) -> None:
    for key in ["run_id", "parameters", "results"]:
        if key not in output:
            raise ValueError(f"Missing top-level key: {key}")

    if not isinstance(output["results"], list):
        raise ValueError("results must be a list")

    for idx, item in enumerate(output["results"], 1):
        for key in ["qid", "question", "retrieved", "answer", "metadata"]:
            if key not in item:
                raise ValueError(f"Result {idx} missing key: {key}")

        if not isinstance(item["retrieved"], list):
            raise ValueError(f"Result {idx} retrieved must be a list")

        for ridx, retrieved in enumerate(item["retrieved"], 1):
            for key in ["rank", "doc_name", "page", "metadata"]:
                if key not in retrieved:
                    raise ValueError(
                        f"Result {idx} retrieved {ridx} missing key: {key}"
                    )

            if retrieved["rank"] != ridx:
                raise ValueError(
                    f"Result {idx} retrieved ranks must be consecutive from 1"
                )

    print("Output shape is compatible with sample_queries.json / Task 1 format.")


validate_official_task1_json(rag_output)

Output shape is compatible with sample_queries.json / Task 1 format.


In [15]:
# 查看最终 JSON 的前几个结果
pd.DataFrame(
    [
        {
            "qid": item["qid"],
            "question": item["question"],
            "retrieved": [
                f"{x['doc_name']} p.{x['page']}"
                for x in item["retrieved"]
            ],
            "answer_preview": item["answer"][:300],
        }
        for item in rag_output["results"]
    ]
).head()

,qid,question,retrieved,answer_preview
0,Q1,"Quel est l'objectif du projet ""Beehive"" de la ...",[20260108_NP_Obsdrones_Bulletin-de-veille-n12_...,"### **Projet ""Beehive"" de la Royal Navy**\n\n#..."
1,Q2,Comment l’intégration du drone MQ-9 Reaper dan...,[MQ-9 Reaper _ Ministère des Armées et des Anc...,### **Comparaison de l’intégration du MQ-9 Rea...
2,Q3,Comment les mesures anti-drones proposées en F...,[Journées de Lutte Anti-Drones en Méditerranée...,### **Analyse des tendances mondiales en matiè...
3,Q4,Comment les méthodes de renseignement telles q...,"[IC_OSINT_Strategy.pdf p.4, l16b1454_rapport-i...",### **Intégration de l’OSINT et de l’HUMINT au...
4,Q5,Quels sont les drones capables de transporter ...,[Pôle innovation technique de défense IDEA3 FS...,Voici une synthèse des drones et systèmes capa...
